# Running the IE643 demo

This notebook starts the Gradio interface for the pharmacy-label reconstruction demo.

Before running it, make sure the project folder in Google Drive contains `app/ie643_demo.py` and the required model checkpoints. The notebook installs Gradio, sets the file paths, and launches the interface. Update the paths in the configuration cell if your Drive folder is different.

The model checkpoint files are not included in the GitHub repository because of their size; they need to be available separately in the locations configured below.


In [1]:
# cell 1: mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# cell 2: install gradio (TensorFlow, PyTorch, OpenCV and scikit-image are already in Colab)
!pip -q install gradio
import tensorflow as tf, torch, gradio
print('TF', tf.__version__, '| torch', torch.__version__, '| gradio', gradio.__version__, '| GPU', torch.cuda.is_available())

TF 2.21.0 | torch 2.11.0+cu130 | gradio 6.30.0 | GPU True


In [3]:
# cell 3: paths (edit if your files live elsewhere)
from pathlib import Path
PROJECT   = Path('/content/drive/MyDrive/IE643_Project')
APP       = PROJECT / 'app' / 'ie643_demo.py'
UNET_CKPT = PROJECT / 'app' / 'unet_group_safe_best.keras'                   # your best U-Net checkpoint
RECON_CKPT= PROJECT / 'app' / 'best_ema.pth'                                  # from the Kaggle output
DATA_ROOT = PROJECT / 'dataset416'                                            # clean/ damaged/ masks/
SPLIT_CSV = PROJECT / 'app' / 'dataset_split_v2.csv'                          # optional

for name, p in [('app', APP), ('U-Net', UNET_CKPT), ('recon', RECON_CKPT), ('dataset', DATA_ROOT), ('split csv', SPLIT_CSV)]:
    print(f'{name:10s} {"OK     " if p.exists() else "MISSING"} {p}')

app        OK      /content/drive/MyDrive/IE643_Project/app/ie643_demo.py
U-Net      OK      /content/drive/MyDrive/IE643_Project/app/unet_group_safe_best.keras
recon      OK      /content/drive/MyDrive/IE643_Project/app/best_ema.pth
dataset    MISSING /content/drive/MyDrive/IE643_Project/dataset416
split csv  OK      /content/drive/MyDrive/IE643_Project/app/dataset_split_v2.csv


In [ ]:
# cell 4: launch (keeps running; stop the cell to shut the app down)
!cp "{APP}" /content/ie643_demo.py
args = f'--unet "{UNET_CKPT}" --recon "{RECON_CKPT}" --share'
if DATA_ROOT.exists(): args += f' --data-root "{DATA_ROOT}"'
if SPLIT_CSV.exists(): args += f' --split-csv "{SPLIT_CSV}"'
!python -u /content/ie643_demo.py {args}

[U-Net] exporting Keras weights in a CPU-only helper process ...
[U-Net] loaded /content/drive/MyDrive/IE643_Project/app/unet_group_safe_best.keras | input 256x256 | params 7,771,873 | device cuda | max |torch-keras| = 2.26e-06 (OK) | 21.8s
[Recon] loaded /content/drive/MyDrive/IE643_Project/app/best_ema.pth | epoch 19 | 416px | params 22.29M | device cuda | warm-up 1.2s
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://6064d53217613cd989.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
[run] input (256, 256, 3) -> U-Net ...
[run] U-Net done in 0.03s
[run] diffusion DDIM-25 on cuda, mask 8.8% ...
[run] diffusion done in 3.65s
[run] finished: mask 8.85% | diffusion 3.65s
